In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 - Validações de Qualidade
# MAGIC
# MAGIC Objetivo:
# MAGIC - Validar a consistência entre as camadas Bronze, Silver e Gold
# MAGIC - Verificar duplicidades, chaves nulas e relacionamentos quebrados
# MAGIC - Identificar possíveis problemas de qualidade remanescentes
# MAGIC - Gerar evidências para documentação técnica do case

from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
BASE_PATH = "/Volumes/case/default/case_data"

BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)

display(dbutils.fs.ls(GOLD_PATH))

BRONZE_PATH: /Volumes/case/default/case_data/bronze
SILVER_PATH: /Volumes/case/default/case_data/silver
GOLD_PATH: /Volumes/case/default/case_data/gold


path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/gold/dim_canal/,dim_canal/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/dim_cliente/,dim_cliente/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/dim_data/,dim_data/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/dim_produto/,dim_produto/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/dim_regiao/,dim_regiao/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/dim_vendedor/,dim_vendedor/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/fato_entrega/,fato_entrega/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/fato_ocorrencia/,fato_ocorrencia/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/fato_pedido/,fato_pedido/,0,1781210874197
dbfs:/Volumes/case/default/case_data/gold/fato_pedido_item/,fato_pedido_item/,0,1781210874197


In [0]:
# Bronze
bronze_pedidos_cabecalho = spark.read.format("delta").load(f"{BRONZE_PATH}/pedidos_cabecalho")
bronze_pedidos_itens = spark.read.format("delta").load(f"{BRONZE_PATH}/pedidos_itens")
bronze_clientes = spark.read.format("delta").load(f"{BRONZE_PATH}/clientes")
bronze_produtos = spark.read.format("delta").load(f"{BRONZE_PATH}/produtos")
bronze_entregas = spark.read.format("delta").load(f"{BRONZE_PATH}/entregas")
bronze_ocorrencias = spark.read.format("delta").load(f"{BRONZE_PATH}/ocorrencias")

# Silver
silver_pedidos_cabecalho = spark.read.format("delta").load(f"{SILVER_PATH}/pedidos_cabecalho")
silver_pedidos_itens = spark.read.format("delta").load(f"{SILVER_PATH}/pedidos_itens")
silver_clientes = spark.read.format("delta").load(f"{SILVER_PATH}/clientes")
silver_produtos = spark.read.format("delta").load(f"{SILVER_PATH}/produtos")
silver_entregas = spark.read.format("delta").load(f"{SILVER_PATH}/entregas")
silver_ocorrencias = spark.read.format("delta").load(f"{SILVER_PATH}/ocorrencias")

# Gold
dim_cliente = spark.read.format("delta").load(f"{GOLD_PATH}/dim_cliente")
dim_produto = spark.read.format("delta").load(f"{GOLD_PATH}/dim_produto")
dim_canal = spark.read.format("delta").load(f"{GOLD_PATH}/dim_canal")
dim_vendedor = spark.read.format("delta").load(f"{GOLD_PATH}/dim_vendedor")
dim_regiao = spark.read.format("delta").load(f"{GOLD_PATH}/dim_regiao")
dim_data = spark.read.format("delta").load(f"{GOLD_PATH}/dim_data")

fato_pedido = spark.read.format("delta").load(f"{GOLD_PATH}/fato_pedido")
fato_pedido_item = spark.read.format("delta").load(f"{GOLD_PATH}/fato_pedido_item")
fato_entrega = spark.read.format("delta").load(f"{GOLD_PATH}/fato_entrega")
fato_ocorrencia = spark.read.format("delta").load(f"{GOLD_PATH}/fato_ocorrencia")

mart_performance_comercial = spark.read.format("delta").load(f"{GOLD_PATH}/mart_performance_comercial")
mart_performance_produtos = spark.read.format("delta").load(f"{GOLD_PATH}/mart_performance_produtos")
mart_operacional_entregas = spark.read.format("delta").load(f"{GOLD_PATH}/mart_operacional_entregas")
mart_atendimento_ocorrencias = spark.read.format("delta").load(f"{GOLD_PATH}/mart_atendimento_ocorrencias")

print("Tabelas carregadas com sucesso.")

Tabelas carregadas com sucesso.


In [0]:
validacao_contagens = [
    ("pedidos_cabecalho", bronze_pedidos_cabecalho.count(), silver_pedidos_cabecalho.count(), fato_pedido.count()),
    ("pedidos_itens", bronze_pedidos_itens.count(), silver_pedidos_itens.count(), fato_pedido_item.count()),
    ("clientes", bronze_clientes.count(), silver_clientes.count(), dim_cliente.count()),
    ("produtos", bronze_produtos.count(), silver_produtos.count(), dim_produto.count()),
    ("entregas", bronze_entregas.count(), silver_entregas.count(), fato_entrega.count()),
    ("ocorrencias", bronze_ocorrencias.count(), silver_ocorrencias.count(), fato_ocorrencia.count())
]

df_validacao_contagens = spark.createDataFrame(
    validacao_contagens,
    ["entidade", "qtd_bronze", "qtd_silver", "qtd_gold"]
).withColumn(
    "status_validacao",
    F.when(F.col("qtd_silver") <= F.col("qtd_bronze"), "OK")
     .otherwise("VERIFICAR")
)

display(df_validacao_contagens)

entidade,qtd_bronze,qtd_silver,qtd_gold,status_validacao
pedidos_cabecalho,403,403,403,OK
pedidos_itens,995,995,995,OK
clientes,183,183,180,OK
produtos,72,72,71,OK
entregas,322,322,322,OK
ocorrencias,270,270,270,OK


In [0]:
validacao_chaves_nulas = [
    ("fato_pedido", "order_id", fato_pedido.filter(F.col("order_id").isNull()).count()),
    ("fato_pedido", "customer_id", fato_pedido.filter(F.col("customer_id").isNull()).count()),
    ("fato_pedido", "seller_id", fato_pedido.filter(F.col("seller_id").isNull()).count()),
    ("fato_pedido_item", "order_id", fato_pedido_item.filter(F.col("order_id").isNull()).count()),
    ("fato_pedido_item", "product_id", fato_pedido_item.filter(F.col("product_id").isNull()).count()),
    ("fato_entrega", "delivery_id", fato_entrega.filter(F.col("delivery_id").isNull()).count()),
    ("fato_entrega", "order_id", fato_entrega.filter(F.col("order_id").isNull()).count()),
    ("fato_ocorrencia", "ticket_id", fato_ocorrencia.filter(F.col("ticket_id").isNull()).count()),
    ("fato_ocorrencia", "order_id", fato_ocorrencia.filter(F.col("order_id").isNull()).count()),
    ("dim_cliente", "customer_id", dim_cliente.filter(F.col("customer_id").isNull()).count()),
    ("dim_produto", "product_id", dim_produto.filter(F.col("product_id").isNull()).count())
]

df_validacao_chaves_nulas = spark.createDataFrame(
    validacao_chaves_nulas,
    ["tabela", "campo", "qtd_nulos"]
).withColumn(
    "status_validacao",
    F.when(F.col("qtd_nulos") == 0, "OK").otherwise("VERIFICAR")
)

display(df_validacao_chaves_nulas)

tabela,campo,qtd_nulos,status_validacao
fato_pedido,order_id,0,OK
fato_pedido,customer_id,0,OK
fato_pedido,seller_id,0,OK
fato_pedido_item,order_id,0,OK
fato_pedido_item,product_id,0,OK
fato_entrega,delivery_id,0,OK
fato_entrega,order_id,0,OK
fato_ocorrencia,ticket_id,0,OK
fato_ocorrencia,order_id,0,OK
dim_cliente,customer_id,0,OK


In [0]:
def contar_duplicidades(df, chave):
    return (
        df.groupBy(chave)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

validacao_duplicidades = [
    ("dim_cliente", "customer_id", contar_duplicidades(dim_cliente, "customer_id")),
    ("dim_produto", "product_id", contar_duplicidades(dim_produto, "product_id")),
    ("dim_canal", "channel_id", contar_duplicidades(dim_canal, "channel_id")),
    ("dim_vendedor", "seller_id", contar_duplicidades(dim_vendedor, "seller_id")),
    ("fato_pedido", "order_id", contar_duplicidades(fato_pedido, "order_id")),
    ("fato_entrega", "delivery_id", contar_duplicidades(fato_entrega, "delivery_id")),
    ("fato_ocorrencia", "ticket_id", contar_duplicidades(fato_ocorrencia, "ticket_id"))
]

df_validacao_duplicidades = spark.createDataFrame(
    validacao_duplicidades,
    ["tabela", "campo_chave", "qtd_chaves_duplicadas"]
).withColumn(
    "status_validacao",
    F.when(F.col("qtd_chaves_duplicadas") == 0, "OK").otherwise("VERIFICAR")
)

display(df_validacao_duplicidades)

tabela,campo_chave,qtd_chaves_duplicadas,status_validacao
dim_cliente,customer_id,0,OK
dim_produto,product_id,0,OK
dim_canal,channel_id,0,OK
dim_vendedor,seller_id,0,OK
fato_pedido,order_id,3,VERIFICAR
fato_entrega,delivery_id,2,VERIFICAR
fato_ocorrencia,ticket_id,0,OK


In [0]:
pedidos_sem_cliente = (
    fato_pedido.alias("p")
    .join(dim_cliente.alias("c"), F.col("p.customer_id") == F.col("c.customer_id"), "left_anti")
)

pedidos_sem_vendedor = (
    fato_pedido.alias("p")
    .join(dim_vendedor.alias("v"), F.col("p.seller_id") == F.col("v.seller_id"), "left_anti")
)

itens_sem_pedido = (
    fato_pedido_item.alias("i")
    .join(fato_pedido.alias("p"), F.col("i.order_id") == F.col("p.order_id"), "left_anti")
)

itens_sem_produto = (
    fato_pedido_item.alias("i")
    .join(dim_produto.alias("prod"), F.col("i.product_id") == F.col("prod.product_id"), "left_anti")
)

entregas_sem_pedido = (
    fato_entrega.alias("e")
    .join(fato_pedido.alias("p"), F.col("e.order_id") == F.col("p.order_id"), "left_anti")
)

ocorrencias_sem_pedido = (
    fato_ocorrencia.alias("o")
    .join(fato_pedido.alias("p"), F.col("o.order_id") == F.col("p.order_id"), "left_anti")
)

validacao_relacionamentos = [
    ("fato_pedido", "dim_cliente", "customer_id", pedidos_sem_cliente.count()),
    ("fato_pedido", "dim_vendedor", "seller_id", pedidos_sem_vendedor.count()),
    ("fato_pedido_item", "fato_pedido", "order_id", itens_sem_pedido.count()),
    ("fato_pedido_item", "dim_produto", "product_id", itens_sem_produto.count()),
    ("fato_entrega", "fato_pedido", "order_id", entregas_sem_pedido.count()),
    ("fato_ocorrencia", "fato_pedido", "order_id", ocorrencias_sem_pedido.count())
]

df_validacao_relacionamentos = spark.createDataFrame(
    validacao_relacionamentos,
    ["tabela_origem", "tabela_referencia", "campo_relacionamento", "qtd_sem_relacionamento"]
).withColumn(
    "status_validacao",
    F.when(F.col("qtd_sem_relacionamento") == 0, "OK").otherwise("VERIFICAR")
)

display(df_validacao_relacionamentos)

tabela_origem,tabela_referencia,campo_relacionamento,qtd_sem_relacionamento,status_validacao
fato_pedido,dim_cliente,customer_id,8,VERIFICAR
fato_pedido,dim_vendedor,seller_id,0,OK
fato_pedido_item,fato_pedido,order_id,0,OK
fato_pedido_item,dim_produto,product_id,1,VERIFICAR
fato_entrega,fato_pedido,order_id,1,VERIFICAR
fato_ocorrencia,fato_pedido,order_id,0,OK


In [0]:
validacao_valores = [
    (
        "fato_pedido",
        "gross_amount",
        fato_pedido.filter(F.col("gross_amount") < 0).count(),
        "Valor bruto negativo"
    ),
    (
        "fato_pedido",
        "discount_amount",
        fato_pedido.filter(F.col("discount_amount") < 0).count(),
        "Desconto negativo"
    ),
    (
        "fato_pedido",
        "net_amount",
        fato_pedido.filter(F.col("net_amount") < 0).count(),
        "Receita líquida negativa"
    ),
    (
        "fato_pedido_item",
        "quantity",
        fato_pedido_item.filter(F.col("quantity") <= 0).count(),
        "Quantidade menor ou igual a zero"
    ),
    (
        "fato_pedido_item",
        "unit_price",
        fato_pedido_item.filter(F.col("unit_price") < 0).count(),
        "Preço unitário negativo"
    ),
    (
        "fato_entrega",
        "delivery_cost",
        fato_entrega.filter(F.col("delivery_cost") < 0).count(),
        "Custo de entrega negativo"
    ),
    (
        "fato_entrega",
        "delivery_days",
        fato_entrega.filter(F.col("delivery_days") < 0).count(),
        "Data de entrega anterior à data de envio"
    )
]

df_validacao_valores = spark.createDataFrame(
    validacao_valores,
    ["tabela", "campo", "qtd_inconsistencias", "regra_validacao"]
).withColumn(
    "status_validacao",
    F.when(F.col("qtd_inconsistencias") == 0, "OK").otherwise("VERIFICAR")
)

display(df_validacao_valores)

tabela,campo,qtd_inconsistencias,regra_validacao,status_validacao
fato_pedido,gross_amount,0,Valor bruto negativo,OK
fato_pedido,discount_amount,0,Desconto negativo,OK
fato_pedido,net_amount,5,Receita líquida negativa,VERIFICAR
fato_pedido_item,quantity,20,Quantidade menor ou igual a zero,VERIFICAR
fato_pedido_item,unit_price,0,Preço unitário negativo,OK
fato_entrega,delivery_cost,0,Custo de entrega negativo,OK
fato_entrega,delivery_days,0,Data de entrega anterior à data de envio,OK


In [0]:
validacao_datas = [
    (
        "fato_pedido",
        "promised_date < order_date",
        fato_pedido.filter(F.col("promised_date") < F.col("order_date")).count()
    ),
    (
        "fato_entrega",
        "delivered_at < shipped_at",
        fato_entrega.filter(F.col("delivered_at") < F.col("shipped_at")).count()
    ),
    (
        "fato_pedido",
        "order_date nula",
        fato_pedido.filter(F.col("order_date").isNull()).count()
    ),
    (
        "fato_entrega",
        "shipped_at nula",
        fato_entrega.filter(F.col("shipped_at").isNull()).count()
    ),
    (
        "fato_ocorrencia",
        "created_at nula",
        fato_ocorrencia.filter(F.col("created_at").isNull()).count()
    )
]

df_validacao_datas = spark.createDataFrame(
    validacao_datas,
    ["tabela", "regra_validacao", "qtd_inconsistencias"]
).withColumn(
    "status_validacao",
    F.when(F.col("qtd_inconsistencias") == 0, "OK").otherwise("VERIFICAR")
)

display(df_validacao_datas)

tabela,regra_validacao,qtd_inconsistencias,status_validacao
fato_pedido,promised_date < order_date,0,OK
fato_entrega,delivered_at < shipped_at,0,OK
fato_pedido,order_date nula,1,VERIFICAR
fato_entrega,shipped_at nula,1,VERIFICAR
fato_ocorrencia,created_at nula,0,OK


In [0]:
resumo_qualidade = [
    (
        "Contagem entre camadas",
        df_validacao_contagens.filter(F.col("status_validacao") == "VERIFICAR").count()
    ),
    (
        "Chaves nulas",
        df_validacao_chaves_nulas.filter(F.col("status_validacao") == "VERIFICAR").count()
    ),
    (
        "Duplicidades",
        df_validacao_duplicidades.filter(F.col("status_validacao") == "VERIFICAR").count()
    ),
    (
        "Relacionamentos quebrados",
        df_validacao_relacionamentos.filter(F.col("status_validacao") == "VERIFICAR").count()
    ),
    (
        "Valores inválidos",
        df_validacao_valores.filter(F.col("status_validacao") == "VERIFICAR").count()
    ),
    (
        "Datas incoerentes",
        df_validacao_datas.filter(F.col("status_validacao") == "VERIFICAR").count()
    )
]

df_resumo_qualidade = spark.createDataFrame(
    resumo_qualidade,
    ["grupo_validacao", "qtd_regras_com_alerta"]
).withColumn(
    "status_geral",
    F.when(F.col("qtd_regras_com_alerta") == 0, "OK").otherwise("VERIFICAR")
).withColumn(
    "_data_validacao",
    F.current_timestamp()
)

display(df_resumo_qualidade)

grupo_validacao,qtd_regras_com_alerta,status_geral,_data_validacao
Contagem entre camadas,0,OK,2026-06-11T20:55:18.558Z
Chaves nulas,0,OK,2026-06-11T20:55:18.558Z
Duplicidades,2,VERIFICAR,2026-06-11T20:55:18.558Z
Relacionamentos quebrados,3,VERIFICAR,2026-06-11T20:55:18.558Z
Valores inválidos,2,VERIFICAR,2026-06-11T20:55:18.558Z
Datas incoerentes,2,VERIFICAR,2026-06-11T20:55:18.558Z


In [0]:
VALIDACOES_PATH = f"{GOLD_PATH}/validacoes_qualidade"

tabelas_validacao = {
    "validacao_contagens": df_validacao_contagens,
    "validacao_chaves_nulas": df_validacao_chaves_nulas,
    "validacao_duplicidades": df_validacao_duplicidades,
    "validacao_relacionamentos": df_validacao_relacionamentos,
    "validacao_valores": df_validacao_valores,
    "validacao_datas": df_validacao_datas,
    "resumo_qualidade": df_resumo_qualidade
}

for nome_tabela, df in tabelas_validacao.items():
    caminho_delta = f"{VALIDACOES_PATH}/{nome_tabela}"
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(caminho_delta)
    )
    
    print(f"Validação gravada: {nome_tabela} | Linhas: {df.count()} | Caminho: {caminho_delta}")

Validação gravada: validacao_contagens | Linhas: 6 | Caminho: /Volumes/case/default/case_data/gold/validacoes_qualidade/validacao_contagens
Validação gravada: validacao_chaves_nulas | Linhas: 11 | Caminho: /Volumes/case/default/case_data/gold/validacoes_qualidade/validacao_chaves_nulas
Validação gravada: validacao_duplicidades | Linhas: 7 | Caminho: /Volumes/case/default/case_data/gold/validacoes_qualidade/validacao_duplicidades
Validação gravada: validacao_relacionamentos | Linhas: 6 | Caminho: /Volumes/case/default/case_data/gold/validacoes_qualidade/validacao_relacionamentos
Validação gravada: validacao_valores | Linhas: 7 | Caminho: /Volumes/case/default/case_data/gold/validacoes_qualidade/validacao_valores
Validação gravada: validacao_datas | Linhas: 5 | Caminho: /Volumes/case/default/case_data/gold/validacoes_qualidade/validacao_datas
Validação gravada: resumo_qualidade | Linhas: 6 | Caminho: /Volumes/case/default/case_data/gold/validacoes_qualidade/resumo_qualidade


In [0]:
print("Resumo geral de qualidade")
display(df_resumo_qualidade)

print("Validações com alerta - relacionamentos")
display(
    df_validacao_relacionamentos
    .filter(F.col("status_validacao") == "VERIFICAR")
)

print("Validações com alerta - valores")
display(
    df_validacao_valores
    .filter(F.col("status_validacao") == "VERIFICAR")
)

print("Validações com alerta - datas")
display(
    df_validacao_datas
    .filter(F.col("status_validacao") == "VERIFICAR")
)

Resumo geral de qualidade


grupo_validacao,qtd_regras_com_alerta,status_geral,_data_validacao
Contagem entre camadas,0,OK,2026-06-11T20:58:37.735Z
Chaves nulas,0,OK,2026-06-11T20:58:37.735Z
Duplicidades,2,VERIFICAR,2026-06-11T20:58:37.735Z
Relacionamentos quebrados,3,VERIFICAR,2026-06-11T20:58:37.735Z
Valores inválidos,2,VERIFICAR,2026-06-11T20:58:37.735Z
Datas incoerentes,2,VERIFICAR,2026-06-11T20:58:37.735Z


Validações com alerta - relacionamentos


tabela_origem,tabela_referencia,campo_relacionamento,qtd_sem_relacionamento,status_validacao
fato_pedido,dim_cliente,customer_id,8,VERIFICAR
fato_pedido_item,dim_produto,product_id,1,VERIFICAR
fato_entrega,fato_pedido,order_id,1,VERIFICAR


Validações com alerta - valores


tabela,campo,qtd_inconsistencias,regra_validacao,status_validacao
fato_pedido,net_amount,5,Receita líquida negativa,VERIFICAR
fato_pedido_item,quantity,20,Quantidade menor ou igual a zero,VERIFICAR


Validações com alerta - datas


tabela,regra_validacao,qtd_inconsistencias,status_validacao
fato_pedido,order_date nula,1,VERIFICAR
fato_entrega,shipped_at nula,1,VERIFICAR


In [0]:
# MAGIC %md
# MAGIC ## Detalhamento dos relacionamentos com alerta
# MAGIC
# MAGIC As consultas abaixo exibem os registros que não encontraram correspondência nas dimensões ou fatos relacionados.

print("Pedidos sem cliente")
display(pedidos_sem_cliente)

print("Pedidos sem vendedor")
display(pedidos_sem_vendedor)

print("Itens sem pedido")
display(itens_sem_pedido)

print("Itens sem produto")
display(itens_sem_produto)

print("Entregas sem pedido")
display(entregas_sem_pedido)

print("Ocorrências sem pedido")
display(ocorrencias_sem_pedido)

Pedidos sem cliente


order_id,customer_id,seller_id,order_date,order_date_id,promised_date,promised_date_id,status_order,gross_amount,discount_amount,net_amount,payment_priority,payment_source,dias_ate_promessa,fl_cancelado
O00047,C9999,V038,2025-05-24,20250524,2025-06-07,20250607,FATURADO,11809.41,0.0,11809.41,null,null,14,0
O00094,C9999,V004,2025-08-23,20250823,2025-09-04,20250904,EM_SEPARACAO,1365.05,0.0,1365.05,null,null,12,0
O00141,C9999,V033,2025-11-23,20251123,2025-12-06,20251206,null,7188.45,0.0,7188.45,null,null,13,0
O00188,C9999,V037,2025-02-07,20250207,2025-02-08,20250208,EM_SEPARACAO,2752.08,0.0,2752.08,null,null,1,0
O00235,C9999,V032,2025-09-13,20250913,2025-09-16,20250916,EM_SEPARACAO,14933.87,0.0,14933.87,null,null,3,0
O00282,C9999,V022,2025-12-22,20251222,2026-01-04,20260104,CANCELADO,1758.04,0.0,1758.04,null,null,13,1
O00329,C9999,V026,2025-06-16,20250616,2025-06-25,20250625,CANCELADO,9776.83,1585.07,8191.76,null,null,9,1
O00376,C9999,V024,2025-03-16,20250316,2025-03-20,20250320,FATURADO,13925.52,910.53,13014.99,null,null,4,0


Pedidos sem vendedor


order_id,customer_id,seller_id,order_date,order_date_id,promised_date,promised_date_id,status_order,gross_amount,discount_amount,net_amount,payment_priority,payment_source,dias_ate_promessa,fl_cancelado


Itens sem pedido


...

order_id,item_seq,product_id,quantity,unit_price,total_item,valor_calculado_item,dif_total_item,item_status
O00044,3,P8888,10.0,2188.7,21887.0,21887.0,0.0,null


Entregas sem pedido


delivery_id,order_id,carrier_name,delivery_mode,delivery_status,shipped_at,shipped_date_id,delivered_at,delivered_date_id,destination_state,destination_city,delivery_cost,delivery_days,fl_entregue,fl_entrega_atrasada
D00004,O99999,null,Aéreo,null,2025-02-23,20250223,2025-03-05,20250305,SC,Florianópolis,346.48,10,1,0


Ocorrências sem pedido


ticket_id,order_id,event_type,created_at,created_date_id,severity,status_ocorrencia,fl_ocorrencia_aberta


In [0]:
# MAGIC %md
# MAGIC ## Detalhamento dos valores inconsistentes
# MAGIC
# MAGIC Registros com valores negativos ou quantidades inválidas foram mantidos no modelo,
# MAGIC mas sinalizados para análise de qualidade.

print("Pedidos com receita líquida negativa")
display(
    fato_pedido
    .filter(F.col("net_amount") < 0)
    .select(
        "order_id",
        "customer_id",
        "seller_id",
        "order_date",
        "status_order",
        "gross_amount",
        "discount_amount",
        "net_amount"
    )
)

print("Itens com quantidade menor ou igual a zero")
display(
    fato_pedido_item
    .filter(F.col("quantity") <= 0)
    .select(
        "order_id",
        "item_seq",
        "product_id",
        "quantity",
        "unit_price",
        "total_item",
        "item_status"
    )
)

Pedidos com receita líquida negativa


order_id,customer_id,seller_id,order_date,status_order,gross_amount,discount_amount,net_amount
O00034,C0112,V008,2025-01-29,EM_SEPARACAO,240.81,343.6,-102.79
O00045,C0085,V029,2025-03-11,null,200.83,791.99,-591.16
O00236,C0162,V007,2025-12-20,ENTREGUE,994.54,1627.82,-633.28
O00265,C0021,V030,2026-01-03,ENTREGUE,294.98,1842.15,-1547.17
O00334,C0033,V019,2025-02-15,EM_SEPARACAO,220.98,1972.86,-1751.88


Itens com quantidade menor ou igual a zero


order_id,item_seq,product_id,quantity,unit_price,total_item,item_status
O00005,3,P0030,-1.0,391.05,-391.05,CANCELADO
O00074,1,P0009,-1.0,1348.08,-1348.08,ATIVO
O00078,2,P0018,-1.0,814.89,-814.89,ATIVO
O00124,3,P0054,0.0,1932.01,0.0,CANCELADO
O00156,2,P0017,-1.0,888.41,-888.41,ATIVO
O00192,3,P0067,-1.0,1202.27,-1202.27,ATIVO
O00193,1,P0038,-1.0,2959.07,-2959.07,ATIVO
O00202,1,P0004,0.0,255.73,0.0,ATIVO
O00211,3,P0050,-1.0,1844.85,-1844.85,CANCELADO
O00217,2,P0033,0.0,714.74,0.0,ATIVO


In [0]:
# MAGIC %md
# MAGIC ## Detalhamento das datas inconsistentes
# MAGIC
# MAGIC Registros com datas nulas ou incoerentes foram sinalizados para documentação,
# MAGIC pois impactam análises temporais.

print("Pedidos com order_date nula")
display(
    fato_pedido
    .filter(F.col("order_date").isNull())
)

print("Entregas com shipped_at nula")
display(
    fato_entrega
    .filter(F.col("shipped_at").isNull())
)

print("Pedidos com promised_date menor que order_date")
display(
    fato_pedido
    .filter(F.col("promised_date") < F.col("order_date"))
)

print("Entregas com delivered_at menor que shipped_at")
display(
    fato_entrega
    .filter(F.col("delivered_at") < F.col("shipped_at"))
)

Pedidos com order_date nula


order_id,customer_id,seller_id,order_date,order_date_id,promised_date,promised_date_id,status_order,gross_amount,discount_amount,net_amount,payment_priority,payment_source,dias_ate_promessa,fl_cancelado
O00121,C0024,V008,null,null,2025-03-06,20250306,CANCELADO,11794.66,0.0,11794.66,null,null,null,1


Entregas com shipped_at nula


delivery_id,order_id,carrier_name,delivery_mode,delivery_status,shipped_at,shipped_date_id,delivered_at,delivered_date_id,destination_state,destination_city,delivery_cost,delivery_days,fl_entregue,fl_entrega_atrasada
D00021,O00185,EntregaJá,Aéreo,null,null,null,null,null,RJ,Niterói,170.43,null,0,0


Pedidos com promised_date menor que order_date


order_id,customer_id,seller_id,order_date,order_date_id,promised_date,promised_date_id,status_order,gross_amount,discount_amount,net_amount,payment_priority,payment_source,dias_ate_promessa,fl_cancelado


Entregas com delivered_at menor que shipped_at


delivery_id,order_id,carrier_name,delivery_mode,delivery_status,shipped_at,shipped_date_id,delivered_at,delivered_date_id,destination_state,destination_city,delivery_cost,delivery_days,fl_entregue,fl_entrega_atrasada


In [0]:
# MAGIC %md
# MAGIC ## Resumo interpretativo das validações
# MAGIC
# MAGIC As validações indicaram que a maior parte do pipeline foi processada com sucesso,
# MAGIC porém foram identificados pontos de atenção de qualidade:
# MAGIC
# MAGIC - Existem pedidos sem correspondência na dimensão de clientes.
# MAGIC - Existe item de pedido sem correspondência na dimensão de produtos.
# MAGIC - Existe entrega sem correspondência na fato de pedidos.
# MAGIC - Existem pedidos com receita líquida negativa.
# MAGIC - Existem itens com quantidade menor ou igual a zero.
# MAGIC - Existem registros com datas nulas que podem impactar análises temporais.
# MAGIC
# MAGIC Decisão técnica:
# MAGIC Os registros não foram descartados automaticamente. Eles foram preservados nas tabelas analíticas
# MAGIC e sinalizados em tabelas de validação, pois a remoção poderia causar perda de rastreabilidade
# MAGIC e alteração indevida dos indicadores de negócio.